<a href="https://colab.research.google.com/github/alijawad8586/AI-Career-Path-Recommender/blob/main/Ai_cv_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  AI CV ANALYZER – single-cell notebook edition                    ║
# ║  • Ingests a résumé (PDF / DOCX / TXT)                            ║
# ║  • Parses sections, embeds, and scores against a job description  ║
# ╚═══════════════════════════════════════════════════════════════════╝

# ── 1️⃣  Dependencies ────────────────────────────────────────────────
!pip -q install pdfminer.six python-docx PyYAML sentence-transformers \
              scikit-learn spacy docx2txt

# ── 2️⃣  Imports & lightweight assets ────────────────────────────────
import re, itertools, pathlib, json, yaml, torch, textwrap, datetime
from pdfminer.high_level import extract_text as _pdf_text
import docx2txt, spacy, pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.nn.functional import cosine_similarity

# ── 3️⃣  Text extraction ─────────────────────────────────────────────
def extract_text(file_path: pathlib.Path) -> str:
    suf = file_path.suffix.lower()
    if suf == ".pdf":
        return _pdf_text(file_path)
    if suf in {".docx", ".doc"}:
        return docx2txt.process(str(file_path))
    if suf in {".txt", ".md"}:
        return file_path.read_text(encoding="utf-8", errors="ignore")
    raise ValueError(f"Unsupported file type: {suf}")

# ── 4️⃣  Section parser ──────────────────────────────────────────────
_NLP      = spacy.blank("en")
_HEADERS  = {
    "experience": re.compile(r"\b(work|professional)\s+experience\b", re.I),
    "education":  re.compile(r"\beducation\b", re.I),
    "skills":     re.compile(r"\bskills?\b", re.I),
    "projects":   re.compile(r"\bprojects?\b", re.I),
}

def split_sections(raw: str) -> dict[str, str]:
    lines, sec, out = [l.strip() for l in raw.splitlines()], "misc", {}
    for line, nxt in itertools.zip_longest(lines, lines[1:]):
        for name, pat in _HEADERS.items():
            if pat.fullmatch(line.lower()):
                sec = name
                out[sec] = []
                break
        else:
            out.setdefault(sec, []).append(line)
    return {k: "\n".join(v).strip() for k,v in out.items() if v}

# ── 5️⃣  Embedder wrapper ────────────────────────────────────────────
class Embedder:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)
    def encode(self, texts):
        return self.model.encode(texts, convert_to_tensor=True,
                                 normalize_embeddings=True)

# ── 6️⃣  Scoring logic ───────────────────────────────────────────────
class CVScorer:
    def __init__(self, cfg: dict, embedder: Embedder):
        self.cfg, self.emb = cfg, embedder
        self.tfidf = TfidfVectorizer(stop_words="english",
                                     ngram_range=(1,2),
                                     min_df=cfg["tfidf_min_df"])
    def score(self, cv_sections: dict, jd_text: str) -> float:
        cv_text  = " ".join(cv_sections.values()) or " "
        v_cv, v_jd = self.emb.encode([cv_text, jd_text])
        sim = cosine_similarity(v_cv, v_jd, dim=0).item()

        X = self.tfidf.fit_transform([jd_text, cv_text])
        kw_overlap = (X[0].multiply(X[1])).sum()
        kw_norm    = X[0].power(2).sum() ** 0.5
        tfidf_score = kw_overlap / (kw_norm + 1e-9)

        heuristic = 0.0
        for sec, wt in self.cfg["section_weights"].items():
            if sec in cv_sections and re.search(self.cfg["role_regex"],
                                                cv_sections[sec], re.I):
                heuristic += wt
        final = (
            self.cfg["w_semantic"]  * sim +
            self.cfg["w_keywords"]  * tfidf_score +
            self.cfg["w_heuristic"] * heuristic
        )
        return round(final, 4)

# ── 7️⃣  Config (editable inline) ────────────────────────────────────
CFG = yaml.safe_load("""
model_name: sentence-transformers/all-mpnet-base-v2
w_semantic:  0.55
w_keywords:  0.25
w_heuristic: 0.20
tfidf_min_df: 2
role_regex: "\\b(data scientist|ml engineer|ai)\\b"
section_weights:
  experience: 1.0
  skills:     0.7
  projects:   0.5
""")

emb    = Embedder(CFG["model_name"])
scorer = CVScorer(CFG, emb)

# ── 8️⃣  Example run ─────────────────────────────────────────────────
# ↳  Upload or point at your files then adjust the two paths.
from google.colab import files, drive
print("🔼  OPTIONAL: Use the file-picker to upload résumé/JD now.")
files_uploaded = files.upload()          # comment out if you use Drive

# Alternative: drive.mount('/content/drive'); then use Drive paths.
cv_path = next((pathlib.Path(p) for p in files_uploaded if p.lower().endswith((".pdf",".doc",".docx",".txt"))), None)
jd_path = next((pathlib.Path(p) for p in files_uploaded if p.lower().endswith(".txt")), None)

assert cv_path and jd_path, "Need at least one CV (.pdf/.docx/.txt) and one JD (.txt)"

cv_text   = extract_text(cv_path)
jd_text   = jd_path.read_text(encoding="utf-8")
sections  = split_sections(cv_text)
score     = scorer.score(sections, jd_text)

print("\n🎯  MATCH SCORE:", score)
print("📄  Résumé:", cv_path.name, "vs Job-description:", jd_path.name)
print("📅  Run on:", datetime.datetime.now())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔼  OPTIONAL: Use the file-picker to upload résumé/JD now.


In [ ]:
# Run in a new Colab cell
%%bash
git config --global user.email "you@example.com"
git config --global user.name  "Your Name"
repo="https://<TOKEN>@github.com/<user>/<repo>.git"

# Clone, copy files, commit, push
git clone "$repo"
cp requirements.txt cv_extractor.py section_parser.py embedder.py scorer.py <repo>/
cd <repo>
git add .
git commit -m "Add pipeline modules"
git push